# 17 · Ablation cấu hình LoRA trên công thức B3: vị trí gắn (H3) và rank (H4) (W10)

**Vấn đề:** B3 (QLoRA + OCR) dùng LoRA **r = 16 chỉ trên 4 lớp attention** (q, k, v, o) của LLM — cấu hình chọn từ W04, chưa từng kiểm. Hội đồng sẽ hỏi:
- *"QLoRA paper (Dettmers et al., NeurIPS 2023) khuyên gắn LoRA vào **mọi** lớp tuyến tính — sao nhóm chỉ gắn attention?"* → **H3**
- *"r = 16 đã đủ chưa, tăng r có tốt hơn không?"* (LoRA paper, Hu et al., ICLR 2022: r nhỏ thường đủ) → **H4**

**Thiết kế (luật ghi trong `docs/W10/README.md` TRƯỚC khi chạy):** chỉ đổi mục LoRA, mọi thứ khác **y hệt B3 seed 42 (W08)**: tập con 8.000 câu, OCR VietOCR · v1 · 600 · 0, lr 2e-4, 1 epoch, batch hiệu dụng 16, bf16, 4-bit NF4, L4, eval test2000.

| Run | Lớp gắn LoRA (chỉ LLM) | r | α | Tham số train | Kiểm |
|---|---|---|---|---|---|
| B3 (mốc, đã có) | q, k, v, o | 16 | 32 | 7.372.800 | — |
| **A2** | q, k, v, o + gate, up, down | **4** | 8 | 7.483.392 (+1,5%) | H3 — cùng ngân sách tham số |
| **A3** | q, k, v, o | **64** | 128 | 29.491.200 (×4) | H4 — rank |

α = 2r ở mọi run (hệ số scale α/r = 2 không đổi). **Bẫy đã chặn:** vision encoder của Qwen2.5-VL cũng có lớp `gate_proj / up_proj / down_proj`; A2 dùng regex chỉ khớp lớp của LLM, và `train_qlora` **dừng** nếu LoRA lọt vào vision.

| Phần | Việc | Thời gian (L4) |
|---|---|---|
| A2 | train ≈ 3,2 giờ → eval test2000 ≈ 40 phút | ≈ 3,9 giờ |
| A3 | train ≈ 3,0 giờ → eval test2000 ≈ 40 phút | ≈ 3,7 giờ |
| Kết luận | so ghép cặp với B3 + luật H3/H4 | 1 phút |

**Chia phiên:** mỗi run xong là một điểm dừng an toàn. Mở lại notebook, **Run all**: phần đã xong được bỏ qua; train bị ngắt sẽ chạy tiếp từ checkpoint (mất tối đa ≈ 36 phút).

## 1. Thiết lập môi trường

Giống notebook 14 (cùng pin `transformers==5.17.0`, cùng GPU L4 với B3). Dừng hẳn nếu GPU không phải L4 — đổi GPU là đổi điều kiện so sánh.

*Kỳ vọng: commit mới nhất (có notebook 17), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('notebooks/17_lora_ablation_rank_modules_qwen25vl.ipynb'), 'Code cũ -> git pull chưa lấy được commit notebook 17'
assert os.path.exists('src/adaptation/lora_targets.py'), 'Thiếu src/adaptation/lora_targets.py'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B3). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run + kiểm tra trước khi tốn GPU

- Tập con train đúng **sha256** trong manifest W08 (8.000 câu) — cùng dữ liệu với B3.
- Cấu hình OCR đọc từ `ocr_selection.json` (W08), **không gõ tay**; OCR cache đủ ảnh train + test2000.
- Mốc **B3 seed 42** (W08): train_log + predictions test2000 còn trên Drive, EM = 0,4925.
- Bảng số tham số **tính tay** cho từng cấu hình (để đối chiếu sau khi train).
- Selftest 4 script: `train_qlora` (có `lora override OK`), `lora_targets` (có bẫy vision), `lora_ablation` (luật H3/H4), `run_baseline`.

*Kỳ vọng: `sha256 tập con OK (8000 câu, 2670 ảnh)`, `paddledet_vietocr | v1 | 600 | 0.0`, `train 2670/2670 · test 1486/1486`, `B3 seed 42: EM 0.4925 ✓`, bảng 3 dòng tham số (7.372.800 / 7.483.392 / 29.491.200), `✅ selftest OK (4 script)`.*

In [ ]:
import json, hashlib, subprocess
import pandas as pd
from src.data.ocr import default_cache_path, load_ocr_cache, images_of
from src.eval.run_baseline import prepare_images, DATASETS
from src.adaptation.lora_targets import PRESETS, expected_lora_params, preset_target_modules

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
ADP_ROOT = f'{DRIVE_ROOT}/adapters'
SUB = f'{DRIVE_ROOT}/data/{DS}/train_subset8000_byimage_seed42.json'
TEST = f"{DRIVE_ROOT}/{DATASETS[DS]['data_rel']}"
OUT = f'{EXP_ROOT}/W10_lora_ablation_test2000_L4_seed42'
SAVE_STEPS = 100        # checkpoint ~36 phút/lần (không ảnh hưởng chữ ký run)

man = json.load(open('experiments/W08_data_subsets/train_subset8000_byimage_seed42_manifest.json', encoding='utf-8'))
assert hashlib.sha256(open(SUB, 'rb').read()).hexdigest() == man['output_sha256'], 'Tập con train trên Drive khác manifest!'
print(f"sha256 tập con OK ({man['n_questions']} câu, {man['n_images']} ảnh)")

sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'

IMG = prepare_images(DS, DRIVE_ROOT)
c = load_ocr_cache(default_cache_path(DRIVE_ROOT, DS, BACKEND))
tr, te = set(images_of([SUB], IMG)), set(images_of([TEST], IMG))
print(f"train {len(tr & set(c['images']))}/{len(tr)} · test {len(te & set(c['images']))}/{len(te)}")
assert tr <= set(c['images']) and te <= set(c['images']), 'OCR cache thiếu ảnh -> chạy notebook 10'

# Mốc B3 seed 42 (W08) — KHÔNG train lại
B3_TRAIN = 'W08_B3_qlora_ocr_vietocr_v1_r16_qwen25vl_vitextvqa_sub8k_ep1_bf16_L4_seed42'
B3_EVAL = 'W08_B3_eval_qlora_ocr_vietocr_v1_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42'
for f in (f'{EXP_ROOT}/{B3_TRAIN}/train_log.json', f'{EXP_ROOT}/{B3_EVAL}/predictions.json'):
    assert os.path.exists(f), f'Thiếu {f}'
em_b3 = json.load(open(f'{EXP_ROOT}/{B3_EVAL}/metrics.json', encoding='utf-8'))['metrics']['exact_match']
assert abs(em_b3 - 0.4925) < 1e-9, em_b3
print(f'B3 seed 42: EM {em_b3:.4f} ✓')

# Biến thể: chỉ khác mục LoRA. Tên run có vị trí gắn + r (seed 42 = seed của B3).
TAIL = 'qwen25vl_vitextvqa_sub8k_ep1_bf16_L4_seed42'
VARIANTS = {
    'A2': {'targets': 'all_linear_llm', 'r': 4, 'tag': 'alllinear_r4'},
    'A3': {'targets': 'attn', 'r': 64, 'tag': 'r64'},
}
for lab, v in VARIANTS.items():
    v['train'] = f"W10_{lab}_qlora_ocr_vietocr_v1_{v['tag']}_{TAIL}"
    v['eval'] = f"W10_{lab}_eval_qlora_ocr_vietocr_v1_{v['tag']}_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42"
    v['expected_params'] = expected_lora_params(v['r'], PRESETS[v['targets']]['suffixes'])
    v['n_modules'] = 36 * len(PRESETS[v['targets']]['suffixes'])
display(pd.DataFrame([{'run': 'B3 (mốc)', 'lớp': 'q,k,v,o', 'r': 16, 'alpha': 32,
                       'tham số (tính tay)': f"{expected_lora_params(16, PRESETS['attn']['suffixes']):,}"}] +
                     [{'run': lab, 'lớp': ','.join(PRESETS[v['targets']]['suffixes']), 'r': v['r'], 'alpha': 2 * v['r'],
                       'tham số (tính tay)': f"{v['expected_params']:,}"} for lab, v in VARIANTS.items()]))

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook (không chạy eval trên adapter hỏng)
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên')

checks = {'src.adaptation.train_qlora': 'lora override OK', 'src.adaptation.lora_targets': 'lora targets OK',
          'src.eval.lora_ablation': 'lora ablation rules OK', 'src.eval.run_baseline': 'SELFTEST OK'}
for mod, must in checks.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    assert must in o.stdout and 'SELFTEST OK' in o.stdout, (mod, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK (4 script)')

## 3. Hàm chạy một biến thể (định nghĩa, chưa chạy GPU)

Với mỗi biến thể: **train → kiểm log → eval test2000** (có OCR, đúng như eval B3). Bước nào đã xong thì bỏ qua.

Sau khi train, kiểm `train_log.json` **trước** khi eval:
- seed = 42; r, α, `target_modules` đúng biến thể;
- `lora_audit`: mọi lớp LoRA nằm trong LLM (`{'llm': …}`, **không có** `vision`), đúng số lớp;
- số tham số train (PEFT đếm) = số **tính tay** ở mục 2.

*Kỳ vọng: `✅ hàm run_variant sẵn sàng`.*

In [ ]:
def check_train_log(lab):
    v = VARIANTS[lab]
    meta = json.load(open(f"{EXP_ROOT}/{v['train']}/train_log.json", encoding='utf-8'))['meta']
    assert meta['seed'] == 42, f"train_log ghi seed {meta['seed']}"
    assert (meta['lora']['r'], meta['lora']['lora_alpha']) == (v['r'], 2 * v['r']), meta['lora']
    assert meta['lora']['target_modules'] == preset_target_modules(v['targets']), meta['lora']['target_modules']
    a = meta['lora_audit']
    assert a['by_tower'] == {'llm': v['n_modules']}, f"LoRA audit lệch: {a['by_tower']}"
    assert meta['trainable_params'] == v['expected_params'] == a['lora_params_counted'], \
        (meta['trainable_params'], v['expected_params'], a['lora_params_counted'])
    print(f"✔ {lab} train_log: r={v['r']} α={2 * v['r']} | {a['by_tower']} | {meta['trainable_params']:,} tham số "
          f"({meta['trainable_pct']}%) | {meta['cost']['train_runtime_sec'] / 3600:.2f} h | VRAM {meta['cost']['peak_vram_gb']} GB")
    return meta

def run_variant(lab):
    v = VARIANTS[lab]
    if os.path.exists(f"{EXP_ROOT}/{v['train']}/train_log.json"):
        print(f'⏭ {lab} train: đã có -> bỏ qua')
    else:
        run_cmd(f"python -u -m src.adaptation.train_qlora --dataset {DS} --data_file {SUB} --run_name {v['train']} "
                f"--epochs 1 --save_steps {SAVE_STEPS} --lora_targets {v['targets']} --lora_r {v['r']} {OCR_ARGS}")
    check_train_log(lab)
    ad = f"{ADP_ROOT}/{v['train']}/epoch_1"
    assert os.path.isdir(ad), f'Thiếu adapter {ad}'
    if os.path.exists(f"{EXP_ROOT}/{v['eval']}/metrics.json"):
        print(f'⏭ {lab} eval: đã có -> bỏ qua')
    else:
        run_cmd(f"python -u -m src.eval.run_baseline --dataset {DS} --run_name {v['eval']} --adapter {ad} {OCR_ARGS}")
    m = json.load(open(f"{EXP_ROOT}/{v['eval']}/metrics.json", encoding='utf-8'))['metrics']
    print(f"✔ {lab}: EM={m['exact_match']:.4f} ANLS={m['anls']:.4f}  (B3: EM 0.4925 ANLS 0.7038)\n")
print('✅ hàm run_variant sẵn sàng')

## 4. A2 — all-linear r = 4, cùng ngân sách tham số với B3 (≈ 3,9 giờ)

Kiểm **H3**. Ngay sau khi nạp model (≈ 4 phút), log phải có dòng `LoRA audit: 252 lớp {'llm': 252} | … | tham số LoRA 7,483,392 (tính tay 7483392)`. Nếu thấy lỗi "LoRA đã gắn vào … VISION" thì **dừng** và gửi Claude — không sửa tay.

**Giữ máy không ngủ và tab mở.** Bị ngắt thì Run all lại.

*Kỳ vọng: dòng `LoRA audit …` như trên, `== XONG: 500 step …`, `✔ A2 train_log …`, `== KẾT QUẢ`, `✔ A2: EM=…`. **Điểm dừng 1.***

In [ ]:
run_variant('A2')

## 5. A3 — attention r = 64 (≈ 3,7 giờ)

Kiểm **H4**. Log phải có `LoRA audit: 144 lớp {'llm': 144} | … | tham số LoRA 29,491,200 (tính tay 29491200)`.

*Kỳ vọng: như mục 4 nhưng `A3`, 144 lớp. **Điểm dừng 2.***

In [ ]:
run_variant('A3')

## 6. Kết luận theo luật ghi trước

So ghép cặp **B3 → A2** và **B3 → A3** trên cùng 2.000 câu (`compare_runs.py`: McNemar exact trên EM, ΔANLS + CI95 bootstrap), rồi `lora_ablation.py` áp luật (`docs/W10/README.md`):

| Giả thuyết | Kết luận |
|---|---|
| **H3** (A2 vs B3) | EM(A2) > EM(B3) & p < 0,05 → **ỦNG HỘ H3** · EM(A2) < EM(B3) & p < 0,05 → **BÁC BỎ H3** · p ≥ 0,05 → **H3 KHÔNG ĐƯỢC ỦNG HỘ** |
| **H4** (A3 vs B3) | ΔEM > +1 điểm & p < 0,05 → **BÁC BỎ H4** · p ≥ 0,05 hoặc A3 kém hơn có ý nghĩa → **ỦNG HỘ H4** (kèm độ mạnh theo CI95 ΔEM) |

Script cũng **kiểm lại** rằng A2, A3 chỉ khác B3 ở mục LoRA (cùng dữ liệu, OCR, seed, lr, epoch…). Kết quả nào cũng ghi đủ — **không** chạy thêm seed/cấu hình sau khi thấy số.

*Kỳ vọng: 2 dòng so sánh, bảng chi phí 3 run, 2 dòng `➜ H3 …`, `➜ H4 …`, file `lora_ablation_summary.json`.*

In [ ]:
os.makedirs(OUT, exist_ok=True)
def paired(run_a, run_b, la, lb, out):
    if not os.path.exists(out):
        r = subprocess.run(f'python src/eval/compare_runs.py --run_a {EXP_ROOT}/{run_a} --run_b {EXP_ROOT}/{run_b} '
                           f'--label_a {la} --label_b {lb} --out {out}', shell=True, capture_output=True, text=True)
        assert r.returncode == 0, r.stdout + r.stderr
    p = json.load(open(out, encoding='utf-8')); p.pop('only_b_question_ids', None)
    print(f"{la:>3} -> {lb:<3} EM {p['em_a']:.4f} -> {p['em_b']:.4f} | chỉ {la} đúng {p['n_only_a']}, chỉ {lb} đúng {p['n_only_b']}, "
          f"p={p['p_value']:.3g} | ΔANLS {p['anls_diff']:+.4f} CI95 [{p['anls_ci95'][0]:+.4f}; {p['anls_ci95'][1]:+.4f}]")
    return out

pa = {lab: paired(B3_EVAL, v['eval'], 'B3', lab, f'{OUT}/paired_B3_vs_{lab}.json') for lab, v in VARIANTS.items()}
logs = {'B3': f'{EXP_ROOT}/{B3_TRAIN}/train_log.json',
        **{lab: f"{EXP_ROOT}/{v['train']}/train_log.json" for lab, v in VARIANTS.items()}}
run_cmd('python -m src.eval.lora_ablation --paired ' + ' '.join(f'{k}={p}' for k, p in pa.items()) +
        ' --train_log ' + ' '.join(f'{k}={p}' for k, p in logs.items()) + f' --out {OUT}/lora_ablation_summary.json')

# Dung lượng adapter (ý nghĩa efficiency: file phải tải kèm khi deploy)
def adapter_mb(run):
    f = f'{ADP_ROOT}/{run}/epoch_1/adapter_model.safetensors'
    return round(os.path.getsize(f) / 1024 ** 2, 1) if os.path.exists(f) else None
s = json.load(open(f'{OUT}/lora_ablation_summary.json', encoding='utf-8'))
display(pd.DataFrame(s['table']))
size = {'B3': adapter_mb(B3_TRAIN), **{lab: adapter_mb(v['train']) for lab, v in VARIANTS.items()}}
display(pd.DataFrame([{**c, 'adapter_MB': size.get(c['run'])} for c in s['costs']]))
s['adapter_mb'] = size; s['gpu'] = GPU
json.dump(s, open(f'{OUT}/lora_ablation_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

## 7. Tự trả máy L4 khi chạy xong

Kết quả đã nằm trên Drive. Output phía trên vẫn hiện trong tab sau khi trả máy. Notebook dừng sớm vì lỗi thì máy **không** tự trả → tự bấm **Runtime → Disconnect and delete runtime** sau khi đọc lỗi.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy L4 …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy L4 để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 8. ⏸ DỪNG

Gửi Claude output **mục 2, 4, 5 và 6** (ít nhất mục 6 + các dòng `LoRA audit` và `== XONG`). Nếu tab đã đóng mất output: không sao — mọi kết quả nằm trong `experiments/W10_*` trên Drive, Cursor chép và kiểm lại được.